In [1]:
import pandas as pd
import numpy as np

test = pd.read_csv("test.csv")
test.head()
test.shape

(1459, 80)

In [2]:
train= pd.read_csv("train.csv")
train.head()
train.shape

(1460, 81)

In [3]:
# %pip install ydata-profiling

In [5]:
from ydata_profiling import data_profiling
data = pd.concat([train, test], ignore_index=True)
#profile = ProfileReport(data, title="EDA Report", explorative=True)
#profile.to_file("EDA_Report1.html")

In [ ]:
## concat both and fill NaN 

In [6]:
test["SalePrice"] = -1
data = pd.concat([train, test], ignore_index=True)

In [7]:
missing = data.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print(missing)

PoolQC          2909
MiscFeature     2814
Alley           2721
Fence           2348
MasVnrType      1766
FireplaceQu     1420
LotFrontage      486
GarageFinish     159
GarageQual       159
GarageCond       159
GarageYrBlt      159
GarageType       157
BsmtExposure      82
BsmtCond          82
BsmtQual          81
BsmtFinType2      80
BsmtFinType1      79
MasVnrArea        23
MSZoning           4
BsmtFullBath       2
BsmtHalfBath       2
Functional         2
Utilities          2
GarageArea         1
GarageCars         1
Electrical         1
KitchenQual        1
TotalBsmtSF        1
BsmtUnfSF          1
BsmtFinSF2         1
BsmtFinSF1         1
Exterior2nd        1
Exterior1st        1
SaleType           1
dtype: int64


In [8]:
mode_cols = ["SaleType", "Exterior1st", "Exterior2nd", "Electrical", "KitchenQual", "Functional", "Utilities", "MSZoning"]

none_cols = ["PoolQC", "Alley", "GarageType", "GarageFinish", "GarageQual", "GarageCond","MiscFeature", "Fence", "FireplaceQu", "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2"]

In [9]:
data[none_cols] = data[none_cols].fillna("None")
data[mode_cols] = data[mode_cols].fillna(data[mode_cols].mode().iloc[0])

In [10]:
data.loc[data["TotalBsmtSF"] == 0,
         ["BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF",
          "BsmtFullBath", "BsmtHalfBath"]] = 0

In [11]:
data.loc[data["BsmtQual"] == "None" ,
         ["BsmtFinSF1","TotalBsmtSF", "BsmtFinSF2", "BsmtUnfSF",
          "BsmtFullBath", "BsmtHalfBath"]] = 0

In [12]:
data.loc[data["GarageType"] == "None",
         ["GarageArea","GarageCars"]] = 0

In [13]:
((data["MasVnrType"].isnull()) & (data["MasVnrArea"].isnull())).sum()

np.int64(23)

In [14]:
mask = data["MasVnrArea"].isnull() | (data["MasVnrArea"] == 0)
data.loc[mask, "MasVnrArea"] = 0
data.loc[mask, "MasVnrType"] = "None"

In [15]:
data.loc[
    data["MasVnrType"].isnull() & (data["MasVnrArea"] > 0),
    "MasVnrType"
] = data["MasVnrType"].mode()[0]

In [16]:
((data["GarageType"] == "None") & (data["GarageArea"].isnull())).sum()

np.int64(0)

In [17]:
data[data["GarageCars"].isnull()][["GarageType", "GarageArea", "GarageCars"]]

,GarageType,GarageArea,GarageCars
2576,Detchd,NaN,NaN


In [18]:
data["GarageCars"] = data["GarageCars"].fillna(data["GarageCars"].median())
data["GarageArea"] = data["GarageArea"].fillna(data["GarageArea"].median())

In [19]:
data["LotFrontage"] = data["LotFrontage"].fillna(data["LotFrontage"].median())

In [20]:
data.loc[data["GarageType"] == "None", "GarageYrBlt"] = 0

In [21]:
data["GarageYrBlt"] = data["GarageYrBlt"].fillna(data["GarageYrBlt"].median())

In [22]:
missing = data.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print(missing)
## missing evaluated

Series([], dtype: int64)


In [ ]:
## Handling categorial features

In [23]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

In [24]:
cat_cols = data.select_dtypes(include="object").columns
print(cat_cols)

Index(['MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities',
       'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2',
       'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st',
       'Exterior2nd', 'MasVnrType', 'ExterQual', 'ExterCond', 'Foundation',
       'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
       'Heating', 'HeatingQC', 'CentralAir', 'Electrical', 'KitchenQual',
       'Functional', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual',
       'GarageCond', 'PavedDrive', 'PoolQC', 'Fence', 'MiscFeature',
       'SaleType', 'SaleCondition'],
      dtype='object')


In [25]:
ordinal_cols = ["LotShape", "Utilities", "LandSlope", "ExterQual", "ExterCond", "BsmtQual","Fence", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2", "HeatingQC", "KitchenQual", "Functional", "FireplaceQu", "GarageFinish", "GarageQual", "GarageCond", "PavedDrive", "PoolQC"]

nominal_cols = ["MSZoning", "Street", "Alley", "LandContour", "LotConfig", "Neighborhood","MiscFeature", "Condition1", "Condition2", "BldgType", "HouseStyle", "RoofStyle", "RoofMatl", "Exterior1st", "Exterior2nd", "MasVnrType", "Foundation", "Heating", "CentralAir", "Electrical", "GarageType", "SaleType", "SaleCondition"]

In [26]:
ordinal_encoder = OrdinalEncoder()
ohe_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
data[ordinal_cols] = ordinal_encoder.fit_transform(data[ordinal_cols])

encoded = ohe_encoder.fit_transform(data[nominal_cols])
encoded = pd.DataFrame(
    encoded,
    columns=ohe_encoder.get_feature_names_out(nominal_cols),
    index=data.index
)

data = data.drop(columns=nominal_cols)
data = pd.concat([data, encoded], axis=1)

In [27]:
data.shape

(2919, 227)

In [28]:
df1 = data[data["SalePrice"] != -1].copy()
df2 = data[data["SalePrice"] == -1].drop(columns="SalePrice").copy()

X = df1.drop(columns="SalePrice")
y = df1["SalePrice"]

from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split( X, y, test_size=0.2, random_state=42)
## Not my idea but to calculate accuracy this is best

In [29]:
X_train.shape

(1168, 226)

In [30]:
## numerical vals scaling

In [31]:
num_cols = X_train.select_dtypes(include=["int64", "float64"]).columns

In [33]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_val[num_cols] = scaler.transform(X_val[num_cols])

df2[num_cols] = scaler.transform(df2[num_cols])

In [34]:
## linear regression

In [35]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_val)

print("RMSE:", np.sqrt(mean_squared_error(y_val, y_pred)))
print("MAE :", mean_absolute_error(y_val, y_pred))
print("R2  :", r2_score(y_val, y_pred))

RMSE: 29892.215457096303
MAE : 19280.932898655516
R2  : 0.8835063176196845


In [36]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(lr,X,y,cv=5,scoring="neg_root_mean_squared_error")

print("CV RMSE:", -cv_scores)
print("Mean CV RMSE:", -cv_scores.mean())

CV RMSE: [29900.44501529 34106.97127609 37777.46943569 26537.70859811
 49745.67212602]
Mean CV RMSE: 35613.653290240596


In [ ]:
## Check for ridge and lasso

In [37]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score
import numpy as np

ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)

y_pred = ridge.predict(X_val)

print("RMSE:", np.sqrt(mean_squared_error(y_val, y_pred)))
print("MAE :", mean_absolute_error(y_val, y_pred))
print("R2  :", r2_score(y_val, y_pred))

cv = cross_val_score(ridge, X, y, cv=5, scoring="neg_root_mean_squared_error")
print("Mean CV RMSE:", -cv.mean())

RMSE: 29843.66199129471
MAE : 19227.995858516442
R2  : 0.8838844480673962
Mean CV RMSE: 32687.981124619582


In [38]:
from sklearn.linear_model import Lasso

lasso = Lasso(alpha=1.0, max_iter=100000)
lasso.fit(X_train, y_train)

y_pred = lasso.predict(X_val)

print("RMSE:", np.sqrt(mean_squared_error(y_val, y_pred)))
print("MAE :", mean_absolute_error(y_val, y_pred))
print("R2  :", r2_score(y_val, y_pred))

cv = cross_val_score(lasso, X, y, cv=5, scoring="neg_root_mean_squared_error")
print("Mean CV RMSE:", -cv.mean())

RMSE: 29831.020231038656
MAE : 19200.428855606897
R2  : 0.8839828002111817
Mean CV RMSE: 34848.1259231539


In [40]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import cross_val_score
rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_val)

print("RMSE:", np.sqrt(mean_squared_error(y_val, y_pred)))
print("MAE:", mean_absolute_error(y_val, y_pred))
print("R2:", r2_score(y_val, y_pred))

RMSE: 28908.03581792097
MAE: 17455.721352739725
R2: 0.8910509785011267


In [ ]:
## n_estimaters=100 , maxdepth=None , min_samples_split=2 , min_samples_leaf=1 ,  max_features=None

In [42]:
rf = RandomForestRegressor(n_estimators=100, random_state=42, bootstrap=False)
cv = cross_val_score(lasso, X, y, cv=5, scoring="neg_root_mean_squared_error")
print("Mean CV RMSE:", -cv.mean())

Mean CV RMSE: 34848.1259231539


In [43]:
pip install xgboost

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [44]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import cross_val_score

xgb = XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=3, subsample=0.8, colsample_bytree=0.8, random_state=42)
xgb.fit(X_train, y_train)
y_pred = xgb.predict(X_val)

print("RMSE:", np.sqrt(mean_squared_error(y_val, y_pred)))
print("MAE:", mean_absolute_error(y_val, y_pred))
print("R2:", r2_score(y_val, y_pred))

cv = cross_val_score(xgb, X, y, cv=5, scoring="neg_root_mean_squared_error", n_jobs=-1)
print("Mean CV RMSE:", -cv.mean())

RMSE: 25238.23385262923
MAE: 15605.107421875
R2: 0.9169567823410034
Mean CV RMSE: 26148.686328125


In [45]:
xgb = XGBRegressor(
    n_estimators=700,
    learning_rate=0.1,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.6,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

xgb.fit(X_train, y_train)
y_pred = xgb.predict(X_val)
print("RMSE:", np.sqrt(mean_squared_error(y_val, y_pred)))
print("MAE:", mean_absolute_error(y_val, y_pred))
print("R2:", r2_score(y_val, y_pred))

cv = cross_val_score(
    xgb,
    X,
    y,
    cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

print("Mean CV RMSE:", -cv.mean())

RMSE: 23968.212282103977
MAE: 15047.0751953125
R2: 0.9251042008399963
Mean CV RMSE: 25489.833984375


In [46]:
test_pred = xgb.predict(df2)
submission = pd.DataFrame({
    "Id": test["Id"],
    "SalePrice": test_pred
})

In [47]:
submission.to_csv("submission.csv", index=False)

In [48]:
print(submission.head())
print(submission.columns)

     Id      SalePrice
0  1461  126282.570312
1  1462  172971.562500
2  1463  186293.687500
3  1464  188554.734375
4  1465  194085.718750
Index(['Id', 'SalePrice'], dtype='object')


In [49]:
submission.shape
submission.columns

Index(['Id', 'SalePrice'], dtype='object')